# ============================================
# MODULE 2: MODEL EVALUATION AND VALIDATION
# ============================================
#
# Learning Objectives:
# - Master different evaluation metrics for regression and classification
# - Understand confusion matrices, ROC curves, and precision-recall curves
# - Implement cross-validation strategies for robust model assessment
# - Perform hyperparameter tuning using grid search and random search
# - Diagnose overfitting and underfitting
# - Apply learning curves to assess model performance
#
# Real-World Application:
# In power systems, deploying an ML model with poor evaluation can be catastrophic:
# - False negatives in fault detection can lead to equipment damage
# - Overconfident load forecasts cause generation shortfalls or excess costs
# - Models that overfit training data fail in production
# Proper model evaluation ensures reliable, safe deployment of ML systems.
#
# Estimated Time: 4-5 hours
# ============================================

## Section 1: Import Libraries and Setup

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import model selection and validation tools
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    cross_validate,
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    GridSearchCV,
    RandomizedSearchCV,
    learning_curve,
    validation_curve
)

# Import ML algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Import preprocessing
from sklearn.preprocessing import StandardScaler

# Import regression metrics
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
    explained_variance_score
)

# Import classification metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    matthews_corrcoef,
    cohen_kappa_score
)

# Import warnings
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## Section 2: Generate Power Systems Dataset

In [ ]:
# Generate comprehensive power systems dataset
np.random.seed(42)

n_records = 3000
date_range = pd.date_range(start='2023-01-01', periods=n_records, freq='H')

# Extract temporal features
hours = date_range.hour
day_of_week = date_range.dayofweek
day_of_year = date_range.dayofyear

# Generate realistic load patterns
daily_pattern = 100 + 50 * np.sin((hours - 6) * np.pi / 12)
weekly_pattern = np.where(day_of_week < 5, 1.0, 0.85)
seasonal_pattern = 1.0 + 0.15 * np.cos((day_of_year - 15) * 2 * np.pi / 365)
load_mw = daily_pattern * weekly_pattern * seasonal_pattern + np.random.normal(0, 8, n_records)

# Generate correlated features
temperature_c = 15 + 12 * np.sin((day_of_year - 80) * 2 * np.pi / 365) + \
                5 * np.sin((hours - 14) * np.pi / 12) + np.random.normal(0, 2, n_records)
voltage_kv = 230 - (load_mw - load_mw.mean()) * 0.015 + np.random.normal(0, 1.5, n_records)
power_factor = np.random.uniform(0.88, 0.96, n_records)
current_a = (load_mw * 1000) / (np.sqrt(3) * voltage_kv * power_factor) + np.random.normal(0, 15, n_records)
frequency_hz = 60.0 + (load_mw - load_mw.mean()) * 0.0001 + np.random.normal(0, 0.015, n_records)

# Create binary classification target (equipment health)
# 0: Healthy, 1: Faulty
# Faults more likely with extreme voltage, frequency, or temperature
fault_probability = 0.05  # Base 5% fault rate
fault_score = (
    np.abs(voltage_kv - 230) / 10 +  # Voltage deviation
    np.abs(frequency_hz - 60) * 20 +  # Frequency deviation  
    np.abs(temperature_c - 25) / 15 +  # Temperature deviation
    (load_mw - load_mw.mean()) / 30  # High load stress
)
fault_probability_array = fault_probability + fault_score * 0.1
fault_probability_array = np.clip(fault_probability_array, 0, 0.4)
equipment_fault = (np.random.random(n_records) < fault_probability_array).astype(int)

# Create DataFrame
df = pd.DataFrame({
    'timestamp': date_range,
    'hour': hours,
    'day_of_week': day_of_week,
    'is_weekend': (day_of_week >= 5).astype(int),
    'temperature_c': temperature_c,
    'load_mw': load_mw,
    'voltage_kv': voltage_kv,
    'current_a': current_a,
    'frequency_hz': frequency_hz,
    'power_factor': power_factor,
    'equipment_fault': equipment_fault
})

# Add cyclical features
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

print(f"Generated {len(df)} power system records")
print(f"\nEquipment Fault Distribution:")
print(f"  Healthy: {(df['equipment_fault']==0).sum()} ({(df['equipment_fault']==0).sum()/len(df)*100:.1f}%)")
print(f"  Faulty: {(df['equipment_fault']==1).sum()} ({(df['equipment_fault']==1).sum()/len(df)*100:.1f}%)")

## Section 3: Regression Metrics Deep Dive

Understanding different metrics for evaluating regression models.

In [ ]:
# Prepare regression data (predict load)
feature_cols_reg = ['hour', 'day_of_week', 'is_weekend', 'temperature_c',
                   'voltage_kv', 'frequency_hz', 'hour_sin', 'hour_cos']

X_reg = df[feature_cols_reg]
y_reg = df['load_mw']

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Scale features
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

# Train a Random Forest model
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
rf_reg.fit(X_train_reg_scaled, y_train_reg)

# Make predictions
y_train_pred = rf_reg.predict(X_train_reg_scaled)
y_test_pred = rf_reg.predict(X_test_reg_scaled)

print("Random Forest Regressor trained")
print(f"Training samples: {len(X_train_reg)}")
print(f"Testing samples: {len(X_test_reg)}")

In [ ]:
# Calculate comprehensive regression metrics

print("="*80)
print("COMPREHENSIVE REGRESSION METRICS")
print("="*80)

# Mean Squared Error (MSE)
# Average of squared differences between predicted and actual
# Penalizes large errors more heavily (squared term)
# Units: (target unit)²
mse_train = mean_squared_error(y_train_reg, y_train_pred)
mse_test = mean_squared_error(y_test_reg, y_test_pred)

# Root Mean Squared Error (RMSE)
# Square root of MSE
# Same units as target variable
# Most commonly used metric for regression
rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)

# Mean Absolute Error (MAE)
# Average of absolute differences
# Less sensitive to outliers than RMSE
# Same units as target
mae_train = mean_absolute_error(y_train_reg, y_train_pred)
mae_test = mean_absolute_error(y_test_reg, y_test_pred)

# R² Score (Coefficient of Determination)
# Proportion of variance in target explained by features
# Range: -∞ to 1 (1 is perfect, 0 is baseline, <0 is worse than baseline)
# Unitless metric
r2_train = r2_score(y_train_reg, y_train_pred)
r2_test = r2_score(y_test_reg, y_test_pred)

# Mean Absolute Percentage Error (MAPE)
# Percentage error - good for comparing across different scales
# Can be undefined if actual values are zero
mape_train = mean_absolute_percentage_error(y_train_reg, y_train_pred) * 100
mape_test = mean_absolute_percentage_error(y_test_reg, y_test_pred) * 100

# Explained Variance Score
# Similar to R² but doesn't account for systematic bias
ev_train = explained_variance_score(y_train_reg, y_train_pred)
ev_test = explained_variance_score(y_test_reg, y_test_pred)

# Create results table
metrics_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'R²', 'MAPE (%)', 'Explained Variance'],
    'Training': [mse_train, rmse_train, mae_train, r2_train, mape_train, ev_train],
    'Testing': [mse_test, rmse_test, mae_test, r2_test, mape_test, ev_test],
    'Difference': [
        mse_test - mse_train,
        rmse_test - rmse_train,
        mae_test - mae_train,
        r2_train - r2_test,  # Note: lower is worse for R²
        mape_test - mape_train,
        ev_train - ev_test
    ]
})

print("\n" + metrics_df.to_string(index=False))

print("\n" + "="*80)
print("INTERPRETATION:")
print("="*80)
print(f"• RMSE: {rmse_test:.2f} MW - Average prediction error")
print(f"• MAE: {mae_test:.2f} MW - Average absolute error (robust to outliers)")
print(f"• R²: {r2_test:.4f} - Model explains {r2_test*100:.2f}% of load variance")
print(f"• MAPE: {mape_test:.2f}% - Average percentage error")

# Check for overfitting
if r2_train - r2_test > 0.1:
    print("\n⚠ WARNING: Possible overfitting detected (R² gap > 0.1)")
else:
    print("\n✓ Good generalization (minimal overfitting)")

print("="*80)

In [ ]:
# Visualize regression performance
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Regression Model Evaluation', fontsize=16, fontweight='bold')

# Plot 1: Actual vs Predicted
axes[0, 0].scatter(y_test_reg, y_test_pred, alpha=0.5, s=20, edgecolor='black', linewidth=0.3)
axes[0, 0].plot([y_test_reg.min(), y_test_reg.max()], 
                [y_test_reg.min(), y_test_reg.max()], 
                'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Load (MW)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Predicted Load (MW)', fontsize=11, fontweight='bold')
axes[0, 0].set_title(f'Actual vs Predicted (R² = {r2_test:.4f})', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Residuals (errors)
residuals = y_test_reg - y_test_pred
axes[0, 1].scatter(y_test_pred, residuals, alpha=0.5, s=20, edgecolor='black', linewidth=0.3)
axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted Load (MW)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Residuals (MW)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Error distribution
axes[1, 0].hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[1, 0].axvline(x=residuals.mean(), color='green', linestyle='--', linewidth=2,
                   label=f'Mean: {residuals.mean():.2f} MW')
axes[1, 0].set_xlabel('Prediction Error (MW)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Error Distribution', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Training vs Testing metrics
metrics_comparison = ['RMSE', 'MAE', 'MAPE (%)']
train_values = [rmse_train, mae_train, mape_train]
test_values = [rmse_test, mae_test, mape_test]

x = np.arange(len(metrics_comparison))
width = 0.35

axes[1, 1].bar(x - width/2, train_values, width, label='Training', 
              color='lightblue', edgecolor='black', linewidth=1.5)
axes[1, 1].bar(x + width/2, test_values, width, label='Testing', 
              color='lightcoral', edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('Error', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Training vs Testing Metrics', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(metrics_comparison)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section 4: Classification Metrics Deep Dive

Understanding precision, recall, F1-score, and confusion matrices.

In [ ]:
# Prepare classification data (predict equipment fault)
feature_cols_clf = ['voltage_kv', 'current_a', 'frequency_hz', 'power_factor',
                   'load_mw', 'temperature_c', 'hour', 'is_weekend']

X_clf = df[feature_cols_clf]
y_clf = df['equipment_fault']

# Split data with stratification
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Scale features
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

# Train Random Forest classifier
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_clf.fit(X_train_clf_scaled, y_train_clf)

# Make predictions
y_train_pred_clf = rf_clf.predict(X_train_clf_scaled)
y_test_pred_clf = rf_clf.predict(X_test_clf_scaled)

# Get probability predictions (needed for ROC curve)
y_test_pred_proba = rf_clf.predict_proba(X_test_clf_scaled)[:, 1]

print("Random Forest Classifier trained")
print(f"\nClass distribution in test set:")
print(f"  Healthy: {(y_test_clf==0).sum()} ({(y_test_clf==0).sum()/len(y_test_clf)*100:.1f}%)")
print(f"  Faulty: {(y_test_clf==1).sum()} ({(y_test_clf==1).sum()/len(y_test_clf)*100:.1f}%)")

In [ ]:
# Calculate comprehensive classification metrics

print("="*80)
print("COMPREHENSIVE CLASSIFICATION METRICS")
print("="*80)

# Confusion Matrix Components
# True Positive (TP): Correctly predicted fault
# True Negative (TN): Correctly predicted healthy
# False Positive (FP): Predicted fault but actually healthy (Type I error)
# False Negative (FN): Predicted healthy but actually fault (Type II error)
cm = confusion_matrix(y_test_clf, y_test_pred_clf)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix Components:")
print(f"  True Negatives (TN): {tn}")
print(f"  False Positives (FP): {fp} - Healthy predicted as Faulty")
print(f"  False Negatives (FN): {fn} - Faulty predicted as Healthy ⚠")
print(f"  True Positives (TP): {tp}")

# Accuracy: (TP + TN) / Total
# Overall correctness - can be misleading for imbalanced datasets
accuracy = accuracy_score(y_test_clf, y_test_pred_clf)

# Precision: TP / (TP + FP)
# Of predicted faults, how many are actual faults?
# High precision = few false alarms
precision = precision_score(y_test_clf, y_test_pred_clf)

# Recall (Sensitivity, True Positive Rate): TP / (TP + FN)
# Of actual faults, how many did we detect?
# High recall = few missed faults
recall = recall_score(y_test_clf, y_test_pred_clf)

# Specificity (True Negative Rate): TN / (TN + FP)
# Of actual healthy equipment, how many did we correctly identify?
specificity = tn / (tn + fp)

# F1-Score: Harmonic mean of precision and recall
# Balances precision and recall
# Good when you need balance between false positives and false negatives
f1 = f1_score(y_test_clf, y_test_pred_clf)

# Matthews Correlation Coefficient (MCC)
# Considers all confusion matrix elements
# Range: -1 to 1 (1 is perfect, 0 is random, -1 is perfect inverse)
# Good for imbalanced datasets
mcc = matthews_corrcoef(y_test_clf, y_test_pred_clf)

# Cohen's Kappa
# Agreement between predictions and actual, accounting for chance
# Range: -1 to 1 (1 is perfect agreement)
kappa = cohen_kappa_score(y_test_clf, y_test_pred_clf)

print("\n" + "="*80)
print("Classification Metrics:")
print("="*80)
print(f"Accuracy:     {accuracy:.4f} - Overall correctness")
print(f"Precision:    {precision:.4f} - Predicted faults that are correct")
print(f"Recall:       {recall:.4f} - Actual faults that were detected")
print(f"Specificity:  {specificity:.4f} - Healthy equipment correctly identified")
print(f"F1-Score:     {f1:.4f} - Harmonic mean of precision & recall")
print(f"MCC:          {mcc:.4f} - Overall quality (good for imbalanced)")
print(f"Cohen's Kappa: {kappa:.4f} - Agreement accounting for chance")

print("\n" + "="*80)
print("CRITICAL FOR POWER SYSTEMS:")
print("="*80)
print(f"• False Negatives: {fn} - CRITICAL: Missed faults can cause damage!")
print(f"• False Positives: {fp} - Unnecessary inspections (cost but safe)")
print(f"• Recall: {recall:.4f} - We detect {recall*100:.1f}% of actual faults")
print("="*80)

In [ ]:
# Visualize confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Confusion matrix (counts)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Healthy', 'Faulty'],
            yticklabels=['Healthy', 'Faulty'],
            cbar_kws={'label': 'Count'})
axes[0].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')

# Plot 2: Confusion matrix (percentages)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Greens', ax=axes[1],
            xticklabels=['Healthy', 'Faulty'],
            yticklabels=['Healthy', 'Faulty'],
            cbar_kws={'label': 'Percentage'})
axes[1].set_xlabel('Predicted', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Actual', fontsize=12, fontweight='bold')
axes[1].set_title('Confusion Matrix (Percentages)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print detailed classification report
print("\nDetailed Classification Report:")
print("="*80)
print(classification_report(y_test_clf, y_test_pred_clf, 
                          target_names=['Healthy', 'Faulty']))

## Section 5: ROC Curve and AUC

ROC (Receiver Operating Characteristic) curve shows trade-off between true positive rate and false positive rate.

In [ ]:
# Calculate ROC curve
# ROC curve plots True Positive Rate vs False Positive Rate
# at various classification thresholds

# Calculate ROC curve points
fpr, tpr, thresholds = roc_curve(y_test_clf, y_test_pred_proba)

# Calculate AUC (Area Under Curve)
# AUC = 1.0: Perfect classifier
# AUC = 0.5: Random classifier
# AUC < 0.5: Worse than random
roc_auc = roc_auc_score(y_test_clf, y_test_pred_proba)

# Calculate optimal threshold (maximizes TPR - FPR)
# Youden's J statistic
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print("ROC Curve Analysis")
print("="*80)
print(f"AUC (Area Under Curve): {roc_auc:.4f}")
print(f"Optimal Threshold: {optimal_threshold:.4f}")
print(f"  TPR at optimal: {tpr[optimal_idx]:.4f}")
print(f"  FPR at optimal: {fpr[optimal_idx]:.4f}")
print("\nInterpretation:")
if roc_auc > 0.9:
    print("  Excellent classifier (AUC > 0.9)")
elif roc_auc > 0.8:
    print("  Good classifier (AUC > 0.8)")
elif roc_auc > 0.7:
    print("  Fair classifier (AUC > 0.7)")
else:
    print("  Poor classifier (AUC < 0.7)")
print("="*80)

In [ ]:
# Visualize ROC curve and Precision-Recall curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: ROC Curve
axes[0].plot(fpr, tpr, linewidth=3, label=f'ROC Curve (AUC = {roc_auc:.4f})', color='blue')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier (AUC = 0.5)')
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], s=200, color='red', 
               zorder=5, edgecolor='black', linewidth=2,
               label=f'Optimal Threshold = {optimal_threshold:.3f}')
axes[0].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=12, fontweight='bold')
axes[0].set_title('ROC Curve - Equipment Fault Detection', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Precision-Recall Curve
# Better for imbalanced datasets
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test_clf, y_test_pred_proba)
avg_precision = average_precision_score(y_test_clf, y_test_pred_proba)

axes[1].plot(recall_curve, precision_curve, linewidth=3, 
            label=f'PR Curve (AP = {avg_precision:.4f})', color='green')
# Baseline: proportion of positive class
baseline = (y_test_clf == 1).sum() / len(y_test_clf)
axes[1].plot([0, 1], [baseline, baseline], 'k--', linewidth=2, 
            label=f'Baseline (AP = {baseline:.4f})')
axes[1].set_xlabel('Recall', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Precision', fontsize=12, fontweight='bold')
axes[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nCurve Interpretation:")
print("• ROC Curve: Good for balanced datasets")
print("• Precision-Recall Curve: Better for imbalanced datasets (like fault detection)")
print(f"• Average Precision: {avg_precision:.4f}")

## Section 6: Cross-Validation Strategies

Different cross-validation approaches for different scenarios.

In [ ]:
# Demonstrate different cross-validation strategies

print("="*80)
print("CROSS-VALIDATION STRATEGIES")
print("="*80)

# Strategy 1: K-Fold Cross-Validation
# Splits data into k equal folds
# Trains on k-1 folds, tests on remaining fold
# Repeats k times
# Good for: General purpose, sufficient data
print("\n1. K-Fold Cross-Validation (k=5)")
print("-" * 80)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_kfold = cross_val_score(rf_reg, X_reg, y_reg, cv=kfold, scoring='r2')
print(f"R² scores: {cv_scores_kfold}")
print(f"Mean R²: {cv_scores_kfold.mean():.4f} ± {cv_scores_kfold.std():.4f}")

# Strategy 2: Stratified K-Fold (for classification)
# Ensures each fold has same proportion of each class
# Critical for imbalanced datasets
# Good for: Classification with imbalanced classes
print("\n2. Stratified K-Fold Cross-Validation (k=5)")
print("-" * 80)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_stratified = cross_val_score(rf_clf, X_clf, y_clf, 
                                       cv=stratified_kfold, scoring='f1')
print(f"F1 scores: {cv_scores_stratified}")
print(f"Mean F1: {cv_scores_stratified.mean():.4f} ± {cv_scores_stratified.std():.4f}")

# Strategy 3: Time Series Split
# Respects temporal order of data
# Training set always before test set
# Critical for: Time series data where future shouldn't predict past
print("\n3. Time Series Split (5 splits)")
print("-" * 80)
tscv = TimeSeriesSplit(n_splits=5)
cv_scores_ts = cross_val_score(rf_reg, X_reg, y_reg, cv=tscv, scoring='r2')
print(f"R² scores: {cv_scores_ts}")
print(f"Mean R²: {cv_scores_ts.mean():.4f} ± {cv_scores_ts.std():.4f}")
print("Note: Each split uses more training data (growing window)")

print("\n" + "="*80)
print("CHOOSING THE RIGHT STRATEGY:")
print("="*80)
print("• K-Fold: General purpose, independent samples")
print("• Stratified K-Fold: Classification with imbalanced classes")
print("• Time Series Split: Time-ordered data (load forecasting, price prediction)")
print("="*80)

In [ ]:
# Visualize cross-validation splits
fig, axes = plt.subplots(3, 1, figsize=(16, 10))
fig.suptitle('Cross-Validation Split Strategies', fontsize=16, fontweight='bold')

# Sample data for visualization
n_samples = 100
sample_indices = np.arange(n_samples)

# Plot K-Fold
kf = KFold(n_splits=5)
for i, (train_idx, test_idx) in enumerate(kf.split(sample_indices)):
    axes[0].scatter(train_idx, [i] * len(train_idx), c='blue', marker='s', s=10, label='Train' if i == 0 else '')
    axes[0].scatter(test_idx, [i] * len(test_idx), c='red', marker='s', s=10, label='Test' if i == 0 else '')
axes[0].set_ylabel('Fold', fontsize=11, fontweight='bold')
axes[0].set_title('K-Fold Cross-Validation', fontsize=12, fontweight='bold')
axes[0].set_yticks(range(5))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot Time Series Split
tscv_vis = TimeSeriesSplit(n_splits=5)
for i, (train_idx, test_idx) in enumerate(tscv_vis.split(sample_indices)):
    axes[1].scatter(train_idx, [i] * len(train_idx), c='blue', marker='s', s=10, label='Train' if i == 0 else '')
    axes[1].scatter(test_idx, [i] * len(test_idx), c='red', marker='s', s=10, label='Test' if i == 0 else '')
axes[1].set_ylabel('Fold', fontsize=11, fontweight='bold')
axes[1].set_title('Time Series Split (Growing Window)', fontsize=12, fontweight='bold')
axes[1].set_yticks(range(5))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot comparison of CV scores
cv_comparison = pd.DataFrame({
    'K-Fold': cv_scores_kfold,
    'Time Series': cv_scores_ts
})
cv_comparison.plot(kind='bar', ax=axes[2], color=['steelblue', 'coral'], 
                  edgecolor='black', linewidth=1.5)
axes[2].set_xlabel('Fold Number', fontsize=11, fontweight='bold')
axes[2].set_ylabel('R² Score', fontsize=11, fontweight='bold')
axes[2].set_title('Cross-Validation Scores Comparison', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(range(1, 6), rotation=0)
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section 7: Hyperparameter Tuning

Optimize model parameters using Grid Search and Random Search.

In [ ]:
# Hyperparameter tuning for Random Forest

print("="*80)
print("HYPERPARAMETER TUNING")
print("="*80)

# Define parameter grid for Random Forest
# These are the hyperparameters we want to optimize
param_grid = {
    'n_estimators': [50, 100, 200],        # Number of trees
    'max_depth': [5, 10, 15, None],        # Maximum tree depth
    'min_samples_split': [2, 5, 10],       # Minimum samples to split node
    'min_samples_leaf': [1, 2, 4]          # Minimum samples in leaf
}

print("\nParameter Grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"\nTotal combinations to try: {total_combinations}")

# Use smaller dataset for faster demonstration
X_small = X_reg.iloc[:1000]
y_small = y_reg.iloc[:1000]

print(f"\nUsing {len(X_small)} samples for faster tuning...")

In [ ]:
# Method 1: Grid Search CV
# Exhaustively searches all parameter combinations
# Guaranteed to find best combination in grid
# Slow for large grids

print("\n" + "="*80)
print("METHOD 1: GRID SEARCH CV")
print("="*80)

# Create GridSearchCV object
# cv=3: use 3-fold cross-validation
# n_jobs=-1: use all CPU cores
# verbose=1: show progress
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# Fit grid search
print("\nFitting Grid Search (this may take a minute)...")
grid_search.fit(X_small, y_small)

print("\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest Cross-Validation R² Score: {grid_search.best_score_:.4f}")

# Get top 5 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
top_5 = results_df.nsmallest(5, 'rank_test_score')[['params', 'mean_test_score', 'std_test_score']]
print("\nTop 5 Parameter Combinations:")
print(top_5.to_string(index=False))

In [ ]:
# Method 2: Random Search CV
# Randomly samples parameter combinations
# Faster than grid search
# Good for large parameter spaces

print("\n" + "="*80)
print("METHOD 2: RANDOM SEARCH CV")
print("="*80)

# Define parameter distributions for random sampling
from scipy.stats import randint

param_distributions = {
    'n_estimators': randint(50, 300),           # Randomly sample from 50-300
    'max_depth': [5, 10, 15, 20, None],        # Random choice from list
    'min_samples_split': randint(2, 20),        # Randomly sample from 2-20
    'min_samples_leaf': randint(1, 10)          # Randomly sample from 1-10
}

# Create RandomizedSearchCV
# n_iter: number of random combinations to try
random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions,
    n_iter=20,  # Try 20 random combinations
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\nFitting Random Search (20 iterations)...")
random_search.fit(X_small, y_small)

print("\nBest Parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest Cross-Validation R² Score: {random_search.best_score_:.4f}")

print("\n" + "="*80)
print("COMPARISON:")
print("="*80)
print(f"Grid Search Best Score:   {grid_search.best_score_:.4f}")
print(f"Random Search Best Score: {random_search.best_score_:.4f}")
print("\nNote: Random search is faster and often finds comparable results")
print("="*80)

## Section 8: Learning Curves

Diagnose overfitting and underfitting using learning curves.

In [ ]:
# Generate learning curves
# Learning curves show how performance changes with training set size
# Helps diagnose:
# - Overfitting: Large gap between train and validation scores
# - Underfitting: Both scores are low
# - More data needed: Validation score still improving

print("Generating learning curves...")

# Calculate learning curve for Random Forest
# train_sizes: different training set sizes to try
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    X_reg.iloc[:1500],  # Use subset for speed
    y_reg.iloc[:1500],
    train_sizes=np.linspace(0.1, 1.0, 10),  # 10% to 100% of data
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# Calculate mean and std
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

print("Learning curves generated")

In [ ]:
# Visualize learning curves
plt.figure(figsize=(14, 8))

# Plot training scores
plt.plot(train_sizes, train_mean, linewidth=3, marker='o', markersize=8,
        label='Training Score', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.2, color='blue')

# Plot validation scores
plt.plot(train_sizes, val_mean, linewidth=3, marker='s', markersize=8,
        label='Validation Score', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.2, color='red')

plt.xlabel('Training Set Size', fontsize=12, fontweight='bold')
plt.ylabel('R² Score', fontsize=12, fontweight='bold')
plt.title('Learning Curves: Random Forest Regressor', fontsize=14, fontweight='bold', pad=20)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Add annotations
gap = train_mean[-1] - val_mean[-1]
plt.annotate(f'Gap: {gap:.4f}\n(Train - Val)',
            xy=(train_sizes[-1], (train_mean[-1] + val_mean[-1])/2),
            xytext=(train_sizes[-3], (train_mean[-1] + val_mean[-1])/2),
            arrowprops=dict(arrowstyle='->', color='green', lw=2),
            fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nLearning Curve Interpretation:")
print("="*80)
print(f"Final Training Score: {train_mean[-1]:.4f}")
print(f"Final Validation Score: {val_mean[-1]:.4f}")
print(f"Gap (overfitting indicator): {gap:.4f}")

if gap > 0.1:
    print("\n⚠ Significant overfitting detected (gap > 0.1)")
    print("  Recommendations:")
    print("  - Increase regularization (lower max_depth, increase min_samples)")
    print("  - Get more training data")
    print("  - Reduce model complexity")
elif gap > 0.05:
    print("\n⚡ Moderate overfitting (gap 0.05-0.1)")
    print("  Consider some regularization")
else:
    print("\n✓ Good generalization (gap < 0.05)")

if val_mean[-1] < 0.7:
    print("\n⚠ Underfitting detected (low validation score)")
    print("  Recommendations:")
    print("  - Increase model complexity")
    print("  - Add more features")
    print("  - Try different algorithm")

print("="*80)

## Section 9: Summary and Best Practices

In [ ]:
# Create comprehensive summary
print("="*80)
print("MODEL EVALUATION & VALIDATION - COMPLETE SUMMARY")
print("="*80)

print("\n1. REGRESSION METRICS:")
print("-" * 80)
print("  • RMSE: Primary metric, same units as target")
print("  • MAE: Robust to outliers")
print("  • R²: Proportion of variance explained (0-1)")
print("  • MAPE: Percentage error, scale-independent")

print("\n2. CLASSIFICATION METRICS:")
print("-" * 80)
print("  • Accuracy: Overall correctness (misleading for imbalanced data)")
print("  • Precision: Of predicted positives, how many correct")
print("  • Recall: Of actual positives, how many detected")
print("  • F1-Score: Harmonic mean of precision and recall")
print("  • ROC-AUC: Overall discrimination ability")
print("  • PR-AUC: Better for imbalanced datasets")

print("\n3. CROSS-VALIDATION STRATEGIES:")
print("-" * 80)
print("  • K-Fold: General purpose")
print("  • Stratified K-Fold: Classification with imbalance")
print("  • Time Series Split: Temporal data")

print("\n4. HYPERPARAMETER TUNING:")
print("-" * 80)
print("  • Grid Search: Exhaustive, guaranteed best in grid")
print("  • Random Search: Faster, good for large spaces")

print("\n5. DIAGNOSTICS:")
print("-" * 80)
print("  • Learning Curves: Detect over/underfitting")
print("  • Validation Curves: Tune individual parameters")
print("  • Confusion Matrix: Detailed classification errors")
print("  • ROC/PR Curves: Threshold selection")

print("\n" + "="*80)
print("POWER SYSTEMS SPECIFIC GUIDELINES:")
print("="*80)
print("\n• Load Forecasting (Regression):")
print("  - Use RMSE and MAPE as primary metrics")
print("  - Prefer Time Series CV over K-Fold")
print("  - Monitor residual patterns for time-dependent errors")

print("\n• Fault Detection (Classification):")
print("  - Prioritize Recall (minimize missed faults)")
print("  - Use Stratified K-Fold for imbalanced data")
print("  - Adjust threshold based on cost of false negatives")
print("  - Monitor false negative rate closely")

print("\n• Equipment Health (Classification):")
print("  - Balance Precision and Recall (F1-Score)")
print("  - Use confusion matrix to understand error types")
print("  - Consider cost-sensitive learning")

print("\n" + "="*80)

## What This Means for Electrical Engineers

### Industry Relevance:

1. **Model Validation is Critical**: In power systems, deploying an improperly validated model can have severe consequences:
   - Load forecast errors cost utilities millions in energy markets
   - Missed fault detection leads to equipment damage and safety risks
   - False alarms waste maintenance resources
   - Proper evaluation prevents these costly mistakes

2. **Metric Selection Matters**: Different applications need different metrics:
   - **Load Forecasting**: MAPE is standard (allows comparison across systems)
   - **Fault Detection**: Recall is critical (missing a fault is worse than false alarm)
   - **Equipment Life Prediction**: R² shows how well model explains degradation

3. **Cross-Validation for Reliability**: Single train-test split can be misleading:
   - K-Fold gives robust performance estimate
   - Time Series CV prevents look-ahead bias in forecasting
   - Confidence intervals from CV inform deployment decisions

4. **Hyperparameter Tuning**: Default parameters rarely optimal:
   - Tuning can improve accuracy by 5-15%
   - Grid search finds best combination
   - Random search explores larger parameter spaces
   - Document tuned parameters for reproducibility

### Key Takeaways:

- **Always use appropriate metrics**: Accuracy alone is misleading for imbalanced data
- **Cross-validate everything**: Single split can give overly optimistic results
- **Monitor both train and test**: Large gap indicates overfitting
- **Use learning curves**: Diagnose if you need more data or different model
- **Tune hyperparameters**: Default values are rarely optimal
- **Understand confusion matrix**: Know your error types
- **ROC/PR curves guide thresholds**: Trade-off precision vs recall
- **Time series need special handling**: Use TimeSeriesSplit, not K-Fold

### Common Mistakes:

- Using accuracy for imbalanced classification (fault detection is often <5% faults)
- Not using cross-validation (results don't generalize)
- Fitting scaler on test data (data leakage)
- Using K-Fold for time series (look-ahead bias)
- Ignoring false negatives in fault detection (safety critical)
- Not tuning hyperparameters (leaving performance on table)
- Overfitting to validation set (testing many models on same data)

### Pro Tips:

- **Save your best model**: Use joblib or pickle for deployment
- **Document everything**: Preprocessing, parameters, metrics
- **Use stratification**: Especially important for rare events
- **Set random seeds**: Ensures reproducibility
- **Monitor multiple metrics**: Don't rely on single number
- **Visualize predictions**: Scatter plots reveal patterns in errors
- **Consider domain costs**: Weight false negatives vs false positives appropriately
- **Retrain periodically**: Power systems change over time

### Decision Guide:

**For Load Forecasting:**
- Metrics: RMSE, MAPE, R²
- CV: TimeSeriesSplit
- Tune: Tree depth, number of estimators
- Monitor: Residual patterns by hour/season

**For Fault Detection:**
- Metrics: Recall, F1-Score, ROC-AUC
- CV: Stratified K-Fold
- Tune: Class weights, decision threshold
- Monitor: False negative rate, confusion matrix

**For Equipment Health:**
- Metrics: Balanced accuracy, MCC, Kappa
- CV: Stratified K-Fold
- Tune: Regularization, ensemble size
- Monitor: Per-class performance

### Next Steps:

In Module 3 (Regression), we'll apply these evaluation techniques to build production-quality load forecasting models. We'll see how proper evaluation guides model selection and hyperparameter tuning for real-world power systems applications.

Understanding model evaluation is essential for deploying reliable ML systems in critical infrastructure like power grids.